In [ ]:
import logging
import os
import sys
from pathlib import Path

import numpy as np

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    _WAVELET_BAND_FREQ_RESOLUTION_HZ,
    _wavelet_transform,
    analyzers_to_datasets,
    load_analyzers,
)
from src.definitions.constants import ExperimentNames, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

# Wavelet Data Access (Load or Create)

This notebook prepares wavelet-transformed EEG data for direct use in custom analysis.
It mirrors the internal workflow of `scripts/run_analysis.py --analysis wavelet_power`,
so wavelet files created here can be reused by the script and vice versa.

- Set `REUSE_WAVELETS=True` to load previously stored wavelets when available.
- Set `REUSE_WAVELETS=False` to (re-)compute wavelets and store them.
- Toggle `RUN_BROADBAND` / `RUN_PER_BAND` to control which scopes are computed.

Wavelets are stored under the **same directory structure** as the script:
- Broadband: `WAVELET_DIR/broadband/` — covers the full frequency range spanned
  by `FREQUENCY_BANDS` (DELTA low → GAMMA high, 1 Hz resolution).
- Per-band: `WAVELET_DIR/band_<name>/` — one directory per EEG band.

**Compute-only subset**: set `N_SUBJECTS_SUBSET` to a positive integer to limit
the number of individuals used when computing (not loading) wavelets. This speeds
up interactive exploration. Set to `None` to use all subjects. When loading from
a pre-computed cache (`REUSE_WAVELETS=True`) the full dataset is always retained.

ISC and mean/variance analyses are intentionally skipped.

## Configuration

Defaults match those used by `run_analysis.py` so files can be shared between
the notebook and the CLI script.

In [ ]:
# -- Subject / condition selection -----------------------------------------
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# -- Wavelet transform settings --------------------------------------------
REPRESENTATION = "power"  # "power" or "phase"
# Broadband range derived from the dedicated frequency-band boundaries
# (DELTA low → GAMMA high), using the same 1 Hz resolution as per-band transforms.
WAVELET_FREQ_MIN = min(low for low, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(h for _, h in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / _WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_individuals, n_channels, n_freqs, n_times); only valid for _wavelet_transform, not run_wavelet_workflow

# -- Reuse / compute options -----------------------------------------------
REUSE_WAVELETS = True  # load from cache if available; compute + save otherwise

# -- Subject subset -------------------------------------------------------
# Always limit to the first N individuals.  Set to None to use all subjects.
N_SUBJECTS_SUBSET: int | None = 5
# -- Channel and time subset ----------------------------------------------
# Slice the EEG data before wavelet computation.  The small subset is stored
# in the wavelet cache so subsequent runs simply reload from disk without any
# re-computation.  Set to None to disable the respective slicing.
N_CHANNELS_SUBSET: int | None = 32   # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000   # first N time samples (from ~91 000)

# -- Scope -----------------------------------------------------------------
RUN_BROADBAND = True  # broadband wavelet (all FREQS)
RUN_PER_BAND = True  # per EEG-band wavelets (delta/theta/alpha/beta/gamma)

# -- Storage directory (same default as run_analysis.py --wavelet_data_dir) --
WAVELET_DIR: Path = (
    ProjectPaths.PROCESSED_DATA_DIR / ExperimentNames.PSILO_MUSIC.value / "wavelets"
)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(f"Broadband sub-dir      : {WAVELET_DIR / 'broadband'}")
print(f"Representation         : {REPRESENTATION}")
print(
    f"Frequencies            : {FREQS[0]:.1f}-{FREQS[-1]:.1f} Hz  ({len(FREQS)} steps)"
)
print(f"Reshape to 4D          : {RESHAPE_FREQUENCY_DIM}")
print(
    "Subject subset         : "
    + (f"{N_SUBJECTS_SUBSET} individuals" if N_SUBJECTS_SUBSET is not None else "all individuals")
)
print(
    "Channel subset         : "
    + (f"{N_CHANNELS_SUBSET} channels" if N_CHANNELS_SUBSET is not None else "all channels")
)
print(
    "Time subset            : "
    + (f"{N_TIMES_SUBSET} samples" if N_TIMES_SUBSET is not None else "all samples")
)

## Load source EEG data

In [ ]:
import dataclasses

analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} individuals, {ad.n_features} channels, {ad.n_samples} samples"
    )

## Load or compute wavelet transforms

### Broadband

Stored in `WAVELET_DIR/broadband/` -- same path used by
`run_analysis.py --analysis wavelet_power --wavelet_data_dir <WAVELET_DIR>`.

In [ ]:
broadband_datasets: dict = {}

if RUN_BROADBAND:
    broadband_datasets = _wavelet_transform(
        datasets=datasets,
        freqs=FREQS,
        representation=REPRESENTATION,
        keep_frequency_dim=KEEP_FREQUENCY_DIM,
        reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
        wavelet_dir=WAVELET_DIR / "broadband",
        reuse_wavelets=REUSE_WAVELETS,
    )
    for label, ad in broadband_datasets.items():
        source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
        print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

### Per-band

Each EEG band is stored in its own sub-directory (`WAVELET_DIR/band_<name>/`),
using the same frequency resolution as the script
(`_WAVELET_BAND_FREQ_RESOLUTION_HZ = 1 Hz`).

In [ ]:
band_datasets: dict[str, dict] = {}  # band_name -> {label: AnalysisData}

if RUN_PER_BAND:
    for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
        n_freqs = max(
            2,
            int(round((h_freq - l_freq) / _WAVELET_BAND_FREQ_RESOLUTION_HZ)) + 1,
        )
        band_freqs = np.linspace(l_freq, h_freq, n_freqs)
        band_datasets[band] = _wavelet_transform(
            datasets=datasets,
            freqs=band_freqs,
            representation=REPRESENTATION,
            keep_frequency_dim=KEEP_FREQUENCY_DIM,
            reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
            wavelet_dir=WAVELET_DIR / f"band_{band}",
            reuse_wavelets=REUSE_WAVELETS,
        )
        for label, ad in band_datasets[band].items():
            source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
            print(f"[{band:6s}] {label}: shape={ad.data.shape}  source={source}")

## Access raw wavelet arrays

The cells below expose NumPy arrays for direct use.

- `broadband_arrays[label]` -- shape `(n_individuals, n_channels, n_freqs, n_times)`
  when `RESHAPE_FREQUENCY_DIM=True`.
- `band_arrays[band][label]` -- same shape for each EEG band.

In [ ]:
broadband_arrays = {label: ad.data for label, ad in broadband_datasets.items()}
band_arrays = {
    band: {label: ad.data for label, ad in ds.items()}
    for band, ds in band_datasets.items()
}

# -- Quick summary ---------------------------------------------------------
print("=== Broadband arrays ===")
for label, arr in broadband_arrays.items():
    print(f"  {label}: shape={arr.shape}  dtype={arr.dtype}")

print("\n=== Per-band arrays ===")
for band, arrays in band_arrays.items():
    for label, arr in arrays.items():
        print(f"  [{band:6s}] {label}: shape={arr.shape}  dtype={arr.dtype}")